# CS 361 Homework 2: MNIST with three hidden layers

This notebook compares a two-hidden-layer baseline with a network whose hidden layers have 256, 128, and 64 neurons. Both use the same data split, optimizer, batch size, and epoch limit so the comparison is fair. Run all cells in Google Colab, then capture the training comparison and drawing UI for Gradescope.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

SEED = 361
tf.keras.utils.set_random_seed(SEED)
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
print(x_train.shape, y_train.shape, x_test.shape, y_test.shape)

## Model and backpropagation equations

For one image $x$, use $a_0=x$ and ReLU hidden activations:

$$z_1=W_1a_0+b_1,\quad a_1=\operatorname{ReLU}(z_1)$$
$$z_2=W_2a_1+b_2,\quad a_2=\operatorname{ReLU}(z_2)$$
$$z_3=W_3a_2+b_3,\quad a_3=\operatorname{ReLU}(z_3)$$
$$z_4=W_4a_3+b_4,\quad \hat y=\operatorname{softmax}(z_4)$$

If the expected class is $Y_{exp}$ and $e_{Y_{exp}}$ is its one-hot vector, the cross-entropy loss is

$$L(Y_{exp},\hat y)=-\log(\hat y_{Y_{exp}})=-\sum_{k=0}^{9}(e_{Y_{exp}})_k\log(\hat y_k).$$

Let $\delta_4=\hat y-e_{Y_{exp}}$ and use $\odot$ for elementwise multiplication. Then

$$\delta_3=(W_4^T\delta_4)\odot\operatorname{ReLU}'(z_3),\quad \delta_2=(W_3^T\delta_3)\odot\operatorname{ReLU}'(z_2),\quad \delta_1=(W_2^T\delta_2)\odot\operatorname{ReLU}'(z_1).$$

With learning rate $\eta$ and a batch of size $m$, the updates are

$$W_1\leftarrow W_1-\eta\frac{1}{m}\sum\delta_1a_0^T,\quad b_1\leftarrow b_1-\eta\frac{1}{m}\sum\delta_1$$
$$W_2\leftarrow W_2-\eta\frac{1}{m}\sum\delta_2a_1^T,\quad b_2\leftarrow b_2-\eta\frac{1}{m}\sum\delta_2$$
$$W_3\leftarrow W_3-\eta\frac{1}{m}\sum\delta_3a_2^T,\quad b_3\leftarrow b_3-\eta\frac{1}{m}\sum\delta_3$$
$$W_4\leftarrow W_4-\eta\frac{1}{m}\sum\delta_4a_3^T,\quad b_4\leftarrow b_4-\eta\frac{1}{m}\sum\delta_4.$$

The assignment lists $W_1,b_1,W_2,b_2,W_3$. That convention may count the 10-unit output as layer 3. The requested 256, 128, and 64 hidden widths require a fourth weight matrix for the output, so all four updates are shown here.

In [ ]:
def make_model(hidden_sizes, name):
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(28, 28)),
        tf.keras.layers.Flatten(),
        *[tf.keras.layers.Dense(n, activation='relu') for n in hidden_sizes],
        tf.keras.layers.Dense(10, activation='softmax')
    ], name=name)

def compile_model(model):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'])
    return model

baseline = compile_model(make_model([256, 128], 'two_hidden_layers'))
model_3 = compile_model(make_model([256, 128, 64], 'three_hidden_layers'))
model_3.summary()

In [ ]:
def train_and_measure(model, epochs=10):
    start = time.perf_counter()
    history = model.fit(
        x_train, y_train, validation_split=0.1, epochs=epochs,
        batch_size=128, verbose=2,
        callbacks=[tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=2, restore_best_weights=True)])
    elapsed = time.perf_counter() - start
    test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
    return history, elapsed, test_loss, test_accuracy

history_2, time_2, loss_2, acc_2 = train_and_measure(baseline)
history_3, time_3, loss_3, acc_3 = train_and_measure(model_3)

In [ ]:
comparison = pd.DataFrame([
    {'model': '256-128', 'epochs_run': len(history_2.history['loss']), 'seconds': time_2, 'test_loss': loss_2, 'test_accuracy': acc_2},
    {'model': '256-128-64', 'epochs_run': len(history_3.history['loss']), 'seconds': time_3, 'test_loss': loss_3, 'test_accuracy': acc_3},
])
display(comparison.round(4))

for label, history in [('256-128', history_2), ('256-128-64', history_3)]:
    plt.plot(history.history['val_loss'], marker='o', label=label)
plt.xlabel('Epoch')
plt.ylabel('Validation cross-entropy')
plt.title('Validation error by epoch')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

better = 'did' if loss_3 < loss_2 else 'did not'
faster = 'fewer' if len(history_3.history['loss']) < len(history_2.history['loss']) else 'the same or more'
print(f'The added hidden layer {better} reduce test loss in this run. It used {faster} epochs.')
print('Compare the measured seconds too. A deeper model can reduce loss per epoch, but each epoch usually costs more because it performs extra matrix operations. Results vary with initialization and Colab hardware, so report the values above rather than claiming a universal improvement.')

## Draw a digit and predict it

The canvas is displayed at 280 by 280 pixels and converted to the 28 by 28 grayscale input expected by MNIST. Enter the digit you intended to draw so the UI can display the expected output and cross-entropy error.

In [ ]:
!pip -q install ipycanvas
from ipycanvas import Canvas, hold_canvas
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import output as colab_output
colab_output.enable_custom_widget_manager()

canvas = Canvas(width=280, height=280, sync_image_data=True)
canvas.fill_style = 'black'
canvas.fill_rect(0, 0, 280, 280)
canvas.stroke_style = 'white'
canvas.line_width = 18
canvas.line_cap = 'round'
drawing = False
last = None

def mouse_down(x, y):
    global drawing, last
    drawing, last = True, (x, y)

def mouse_move(x, y):
    global last
    if drawing and last is not None:
        with hold_canvas(canvas):
            canvas.stroke_line(last[0], last[1], x, y)
        last = (x, y)

def mouse_up(x, y):
    global drawing, last
    drawing, last = False, None

canvas.on_mouse_down(mouse_down)
canvas.on_mouse_move(mouse_move)
canvas.on_mouse_up(mouse_up)
expected = widgets.Dropdown(options=list(range(10)), description='Expected:')
predict_button = widgets.Button(description='Predict', button_style='success')
clear_button = widgets.Button(description='Clear')
output = widgets.Output()

def clear_canvas(_):
    canvas.fill_style = 'black'
    canvas.fill_rect(0, 0, 280, 280)
    with output:
        clear_output()

def predict_digit(_):
    rgba = np.asarray(canvas.get_image_data(0, 0, 280, 280).data, dtype=np.uint8).reshape(280, 280, 4)
    gray = rgba[:, :, 0].astype('float32') / 255.0
    image28 = tf.image.resize(gray[..., None], (28, 28), method='area').numpy()[:, :, 0]
    probabilities = model_3.predict(image28[None, ...], verbose=0)[0]
    predicted = int(np.argmax(probabilities))
    expected_digit = int(expected.value)
    error = float(-np.log(np.clip(probabilities[expected_digit], 1e-7, 1.0)))
    with output:
        clear_output()
        print(f'Expected output: {expected_digit}')
        print(f'Predicted output: {predicted}')
        print(f'Cross-entropy error: {error:.6f}')
        print(f'Confidence: {probabilities[predicted]:.2%}')
        plt.figure(figsize=(7, 2.5))
        plt.bar(range(10), probabilities)
        plt.xticks(range(10))
        plt.ylim(0, 1)
        plt.xlabel('Digit')
        plt.ylabel('Probability')
        plt.show()

predict_button.on_click(predict_digit)
clear_button.on_click(clear_canvas)
display(widgets.VBox([canvas, widgets.HBox([expected, predict_button, clear_button]), output]))